In [4]:
# Predict Customer Churn
## Score: 

In [5]:
import numpy as np
import pandas as pd

train = pd.read_csv("playground-series-s6e3/train.csv")
test = pd.read_csv("playground-series-s6e3/test.csv")
sample_submission = pd.read_csv("playground-series-s6e3/sample_submission.csv")

train.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [6]:
def engineer_features(df, total_charges_median=None):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    median_tc = total_charges_median if total_charges_median is not None else df['TotalCharges'].median()
    df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
    df['AvgChargesPerMonth'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    df['MonthlyChargesSqrt'] = np.sqrt(df['MonthlyCharges'])
    df['tenure_squared'] = df['tenure'] ** 2
    df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1))
    df['Contract_MonthToMonth'] = (df['Contract'] == 'Month-to-month').astype(int)
    df['Contract_OneYear'] = (df['Contract'] == 'One year').astype(int)
    df['Contract_TwoYear'] = (df['Contract'] == 'Two year').astype(int)
    df['IsFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
    df['NumStreaming'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
    df['NumStreamingBoth'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)
    addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df['NumAddons'] = sum((df[c] == 'Yes').astype(int) for c in addon_cols)
    df['PaymentElectronic'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['HasDependents'] = (df['Dependents'] == 'Yes').astype(int)
    df['HasPartner'] = (df['Partner'] == 'Yes').astype(int)
    df['SeniorWithShortTenure'] = (df['SeniorCitizen'] == 1) & (df['tenure'] < 12)
    df['SeniorWithShortTenure'] = df['SeniorWithShortTenure'].astype(int)
    df['HighChargeShortTenure'] = (df['MonthlyCharges'] > 70) & (df['tenure'] < 12)
    df['HighChargeShortTenure'] = df['HighChargeShortTenure'].astype(int)
    return df

X = train.drop(columns=['id', 'Churn'])
y = train['Churn'].map({'Yes': 1, 'No': 0})
X_test = test.drop(columns=['id'])
test_ids = test['id']

X = engineer_features(X)
tc_median = X['TotalCharges'].median()
X_test = engineer_features(X_test, total_charges_median=tc_median)

X_encoded = pd.get_dummies(X, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_28916\89863577.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X['TotalCharges'].fillna(X['TotalCharges'].median(), inplace=True)
C:\Users\ol1v3_7dwns5u\AppData\Local\Temp\ipykernel_28916\89863577.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alw

In [7]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [8]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [300, 500],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'scale_pos_weight': [2],
}

xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
grid_search = GridSearchCV(xgb, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_search.fit(X_encoded, y)
best_xgb = grid_search.best_estimator_
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.4f}")

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [9]:
feat_imp = pd.Series(best_xgb.feature_importances_, index=X_encoded.columns).sort_values(ascending=False)
feat_imp.head(15)

Validation ROC-AUC: 0.8949


In [10]:
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

best_xgb.fit(X_encoded, y)
lgb = LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
lgb.fit(X_encoded, y)
cat = CatBoostClassifier(iterations=500, depth=6, learning_rate=0.05, random_state=42, verbose=0)
cat.fit(X_encoded, y)

pred_xgb = best_xgb.predict_proba(X_test_encoded)[:, 1]
pred_lgb = lgb.predict_proba(X_test_encoded)[:, 1]
pred_cat = cat.predict_proba(X_test_encoded)[:, 1]

test_pred_proba = (pred_xgb + pred_lgb + pred_cat) / 3

submission = pd.DataFrame({'id': test_ids, 'Churn': test_pred_proba})
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Churn
0,594194,0.01
1,594195,0.00
2,594196,0.13
3,594197,0.00
4,594198,0.48
